# Analysis: PCA trajectories & dPCA

Load stage-2 model, collect trials, run PCA / dPCA.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src import BioLeakyRNN, CuedTargetWithDistractorsV2
from src.analysis import (
    collect_trials, fit_pca_on_trials, select_trials,
    get_aligned_segments, compute_median_and_band, compute_mean_and_sem,
    dpca_marginals, collect_aligned_hidden_by_label, make_condition_mean_tensor,
)
from src.plotting import (
    plot_pca_trajectories, plot_pca_trajectories_by_outcome,
    plot_pc_timecourses, plot_two_group_pc_timecourses,
    plot_trial_type_panel, plot_dpca_components, plot_dpca_plane,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Load model

In [ ]:
model = BioLeakyRNN(
    input_size=9, hidden_size=128, output_size=2,
    dt=20.0, tau=100.0, activation='softplus', sigma_rec=0.05,
    use_ei=True, exc_ratio=0.7, use_dale=True, mask_seed=42,
).to(device)

ckpt = torch.load('../checkpoints/stage2.pt', weights_only=True)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('Model loaded.')

## Collect trials

In [ ]:
def make_env_stage2():
    return CuedTargetWithDistractorsV2(dt=20, cue_strength=1.0,
                                       p_distractor_trial=0.6, distractor_strength=1.0)

trials = collect_trials(model, make_env_stage2, n_trials=200, device=device)
print(f'Collected {len(trials)} trials')

from collections import Counter
print(Counter(tr['train_outcome'] for tr in trials))

## PCA

In [ ]:
pca, trial_proj, explained = fit_pca_on_trials(trials, n_components=3)
print('Explained variance:', explained)

In [ ]:
plot_pca_trajectories(trials, trial_proj, max_trials=10)

In [ ]:
plot_pca_trajectories_by_outcome(trials, trial_proj, max_per_group=8)

## Event-aligned time courses

In [ ]:
plot_pc_timecourses(
    trials, trial_proj,
    align_key='target_on', window_before=40, window_after=40,
    stat_mode='median', train_outcome='correct',
)

In [ ]:
plot_two_group_pc_timecourses(
    trials, trial_proj,
    align_key='target_on', window_before=40, window_after=40,
    group1_kwargs={'train_outcome': 'correct', 'has_distractors': False},
    group2_kwargs={'train_outcome': 'correct', 'has_distractors': True},
    group1_label='correct / no distractors',
    group2_label='correct / with distractors',
)

## Panel plot: outcomes × CTOA × distractors

In [ ]:
plot_trial_type_panel(
    trials, trial_proj,
    group_specs=[
        {'label': 'correct', 'train_outcome': 'correct'},
        {'label': 'miss',    'train_outcome': 'miss'},
        {'label': 'abort',   'train_outcome': 'abort'},
    ],
    align_key='target_on', window_before=40, window_after=40,
    suptitle='All outcomes | target-locked',
)

## dPCA: CTOA condition

In [ ]:
by_ctoa, rel_time = collect_aligned_hidden_by_label(
    trials,
    label_fn=lambda tr: tr.get('ctoa_bin'),
    align_key='target_on', window_before=40, window_after=40,
)
X_ctoa, ctoa_labels, ctoa_counts = make_condition_mean_tensor(by_ctoa, min_trials=5)
print('CTOA bins:', ctoa_labels, '| counts:', ctoa_counts)

res_ctoa = dpca_marginals(X_ctoa, n_components=3)
res_ctoa['rel_time'] = rel_time
res_ctoa['labels'] = ctoa_labels

In [ ]:
plot_dpca_components(res_ctoa, component_key='Z_time', explained_key='explained_time',
                     title_prefix='time', n_plot=3)
plot_dpca_components(res_ctoa, component_key='Z_cond', explained_key='explained_cond',
                     title_prefix='CTOA', n_plot=3)
plot_dpca_plane(res_ctoa, xlabel='CTOA-dPC1', ylabel='CTOA-dPC2',
                title='CTOA-demixed trajectories')

## dPCA: distractor condition

In [ ]:
by_distr, rel_time = collect_aligned_hidden_by_label(
    trials,
    label_fn=lambda tr: 'with' if tr.get('has_distractors') else 'without',
    align_key='target_on', window_before=40, window_after=40,
)
X_distr, distr_labels, distr_counts = make_condition_mean_tensor(by_distr, min_trials=5)

res_distr = dpca_marginals(X_distr, n_components=3)
res_distr['rel_time'] = rel_time
res_distr['labels'] = distr_labels

plot_dpca_components(res_distr, component_key='Z_cond', explained_key='explained_cond',
                     title_prefix='distractor', n_plot=3)
plot_dpca_plane(res_distr, xlabel='distractor-dPC1', ylabel='distractor-dPC2',
                title='Distractor-demixed trajectories')